In [9]:
import pandas as pd
b = pd.read_csv("../data/processed/batting_stats.csv")
print("batting_stats columns:", len(b.columns))
print("Has role?", "role" in b.columns)
print("Has opener_innings_pct?", "opener_innings_pct" in b.columns)
print("Has modal_position?", "modal_position" in b.columns)
print("Has SR_Death?", "SR_Death" in b.columns)

batting_stats columns: 38
Has role? True
Has opener_innings_pct? True
Has modal_position? True
Has SR_Death? True


In [10]:
import pandas as pd
import numpy as np

batting = pd.read_csv("../data/processed/batting_stats.csv")
bowling = pd.read_csv("../data/processed/bowling_stats.csv")

print("Batters:", len(batting), "| Bowlers:", len(bowling))
print("Batting cols:", len(batting.columns), "| Bowling cols:", len(bowling.columns))

Batters: 186 | Bowlers: 211
Batting cols: 38 | Bowling cols: 33


In [11]:
def classify_role(row):
    if pd.notna(row.get("opener_innings_pct")) and row["opener_innings_pct"] >= 0.5:
        return "Opener"
    if (pd.notna(row.get("modal_position")) and row["modal_position"] <= 7
        and row.get("balls_Death", 0) >= 200
        and row.get("SR_Death", 0) >= 130):
        return "Finisher"
    if (row.get("balls_Middle", 0) >= 300
        and row.get("avg_Middle", 0) >= 30
        and row.get("SR_Middle", 100) < 140):
        return "Anchor"
    if row.get("balls", 0) < 400:
        return "Bowler"
    return "Middle-order"

batting["role"] = batting.apply(classify_role, axis=1)

print("Role distribution:")
print(batting["role"].value_counts().to_string())
print()
print("Legend check (all should be sensible roles):")
legends = ["V Kohli", "DA Warner", "AB de Villiers", "RG Sharma",
           "MS Dhoni", "KL Rahul", "KA Pollard", "AD Russell", "H Klaasen"]
print(batting[batting["striker"].isin(legends)][
    ["striker", "role", "SR_Death", "balls_Death"]
].round(1).to_string(index=False))

Role distribution:
role
Finisher        72
Opener          55
Middle-order    31
Bowler          22
Anchor           6

Legend check (all should be sensible roles):
       striker     role  SR_Death  balls_Death
AB de Villiers Finisher     222.9        838.0
    AD Russell Finisher     195.9        731.0
     DA Warner   Opener     176.9        355.0
     H Klaasen Finisher     192.7        383.0
    KA Pollard Finisher     170.2       1194.0
      KL Rahul   Opener     178.9        507.0
      MS Dhoni Finisher     175.0       1982.0
     RG Sharma Finisher     181.1        843.0
       V Kohli   Opener     188.4        835.0


In [12]:
def robust_minmax(s, lower=0.02, upper=0.98):
    s = s.astype(float)
    lo, hi = s.quantile(lower), s.quantile(upper)
    s = s.clip(lo, hi)
    return (s - s.min()) / (s.max() - s.min())

def phase_sr(row):
    if row["role"] == "Opener":
        return row.get("SR_Powerplay", row["strike_rate"])
    if row["role"] == "Finisher":
        return row.get("SR_Death", row["strike_rate"])
    if row["role"] == "Anchor":
        return row.get("SR_Middle", row["strike_rate"])
    return row["strike_rate"]

batting["phase_sr"] = batting.apply(phase_sr, axis=1).fillna(batting["strike_rate"])
batting["reliability"] = np.minimum(1.0, batting["balls"] / 1000)

batting["n_sr"]      = robust_minmax(batting["phase_sr"].fillna(batting["phase_sr"].median()))
batting["n_avg"]     = robust_minmax(batting["average"].fillna(0))
batting["n_bdry"]    = robust_minmax(batting["boundary_pct"])
batting["n_consist"] = robust_minmax(1 / (1 + batting["std_innings"].fillna(batting["std_innings"].median())))

BAT_WEIGHTS = {"n_sr": 0.35, "n_avg": 0.25, "n_bdry": 0.25, "n_consist": 0.15}
raw = sum(batting[k] * w for k, w in BAT_WEIGHTS.items())
batting["batting_impact"] = (raw * batting["reliability"] * 100).round(2)

print("Top 15 by batting impact:")
print(batting.nlargest(15, "batting_impact")[
    ["striker", "role", "innings", "runs", "strike_rate",
     "phase_sr", "boundary_pct", "batting_impact"]
].round(2).to_string(index=False))

Top 15 by batting impact:
        striker     role  innings  runs  strike_rate  phase_sr  boundary_pct  batting_impact
     AD Russell Finisher      114  2651       174.18     195.9         26.87           74.58
      H Klaasen Finisher       60  2104       166.72     192.7         21.08           74.18
 AB de Villiers Finisher      170  5162       151.69     222.9         19.51           71.96
    Tilak Varma Finisher       65  1858       144.70     198.2         18.22           67.70
        RR Pant Finisher      136  3865       146.79     202.8         20.13           67.68
       N Pooran Finisher      100  2527       164.09     177.6         22.99           64.28
        TM Head   Opener       52  1556       170.05     177.8         27.32           63.44
       SA Yadav Finisher      164  4581       148.59     178.2         21.41           63.35
Abhishek Sharma   Opener       89  2379       171.27     171.9         26.49           63.29
       RK Singh Finisher       62  1394     

In [13]:
bowling["inv_econ"]    = 1 / bowling["economy"]
bowling["inv_bowl_sr"] = 1 / bowling["bowl_sr"].fillna(bowling["bowl_sr"].median())
bowling["reliability"] = np.minimum(1.0, bowling["legal_balls"] / 800)

bowling["n_econ"] = robust_minmax(bowling["inv_econ"].fillna(bowling["inv_econ"].median()))
bowling["n_sr"]   = robust_minmax(bowling["inv_bowl_sr"].fillna(bowling["inv_bowl_sr"].median()))
bowling["n_dot"]  = robust_minmax(bowling["dot_pct"])
bowling["n_wpm"]  = robust_minmax(bowling["wickets_per_match"])

BOWL_WEIGHTS = {"n_sr": 0.35, "n_econ": 0.30, "n_dot": 0.20, "n_wpm": 0.15}
raw = sum(bowling[k] * w for k, w in BOWL_WEIGHTS.items())
bowling["bowling_impact"] = (raw * bowling["reliability"] * 100).round(2)

print("Top 15 by bowling impact:")
print(bowling.nlargest(15, "bowling_impact")[
    ["bowler", "innings", "wickets", "economy", "bowl_sr",
     "dot_pct", "bowling_impact"]
].round(2).to_string(index=False))

Top 15 by bowling impact:
         bowler  innings  wickets  economy  bowl_sr  dot_pct  bowling_impact
     SL Malinga      122      170     7.15     16.6     43.0           84.15
      RJ Harris       37       45     7.55     18.5     46.9           76.18
NM Coulter-Nile       38       48     7.70     17.9     45.2           75.10
       MM Patel       63       74     7.52     18.3     45.2           74.56
       DW Steyn       95       97     6.91     22.4     48.9           72.62
   JR Hazlewood       52       72     8.62     16.3     43.9           72.34
       K Rabada      101      148     8.80     15.5     41.4           72.21
       MA Starc       56       76     8.82     15.7     42.2           70.89
    Imran Tahir       59       82     7.77     16.0     34.3           70.80
        A Nehra       88      106     7.85     18.0     43.4           70.73
      JJ Bumrah      158      187     7.34     19.4     42.2           70.72
       A Kumble       42       45     6.58     21.

In [14]:
# Raise the reliability floor from 800 → 1500 legal balls
bowling["reliability"] = np.minimum(1.0, bowling["legal_balls"] / 1500)

# Rebalance slightly: less SR weight, more economy weight
BOWL_WEIGHTS = {"n_sr": 0.30, "n_econ": 0.30, "n_dot": 0.25, "n_wpm": 0.15}
raw = sum(bowling[k] * w for k, w in BOWL_WEIGHTS.items())
bowling["bowling_impact"] = (raw * bowling["reliability"] * 100).round(2)

print("Top 15 by bowling impact (v2):")
print(bowling.nlargest(15, "bowling_impact")[
    ["bowler", "innings", "wickets", "economy", "bowl_sr",
     "dot_pct", "reliability", "bowling_impact"]
].round(2).to_string(index=False))
print()

# Are the missing spinners there now?
spinners = ["YS Chahal", "DJ Bravo", "Rashid Khan", "R Ashwin",
            "SP Narine", "A Mishra", "PP Chawla", "Imran Tahir"]
sp = bowling[bowling["bowler"].isin(spinners)][
    ["bowler", "innings", "wickets", "economy", "bowl_sr",
     "bowling_impact"]
].sort_values("bowling_impact", ascending=False).round(2)
sp["rank"] = sp["bowling_impact"].rank(ascending=False).astype(int)
print("Where do the spinners rank?")
print(sp.to_string(index=False))

Top 15 by bowling impact (v2):
        bowler  innings  wickets  economy  bowl_sr  dot_pct  reliability  bowling_impact
    SL Malinga      122      170     7.15     16.6     43.0          1.0           83.91
      DW Steyn       95       97     6.91     22.4     48.9          1.0           75.48
     JJ Bumrah      158      187     7.34     19.4     42.2          1.0           71.37
       A Nehra       88      106     7.85     18.0     43.4          1.0           71.19
      K Rabada      101      148     8.80     15.5     41.4          1.0           71.02
     JC Archer       68       84     8.22     18.8     45.0          1.0           68.74
      MM Patel       63       74     7.52     18.3     45.2          0.9           68.29
      M Morkel       70       77     7.69     21.2     45.3          1.0           67.44
     SP Narine      200      207     6.79     22.5     39.9          1.0           67.26
      RP Singh       82       90     7.90     19.7     44.5          1.0       

In [15]:
bat_imp  = batting[["striker", "batting_impact"]].rename(columns={"striker": "player"})
bowl_imp = bowling[["bowler", "bowling_impact"]].rename(columns={"bowler": "player"})

combined = bat_imp.merge(bowl_imp, on="player", how="outer").fillna(0)

primary   = combined[["batting_impact", "bowling_impact"]].max(axis=1)
secondary = combined[["batting_impact", "bowling_impact"]].min(axis=1)
combined["combined_impact"] = (primary + 0.5 * secondary).round(2)

role_info = batting[["striker", "role", "innings"]].rename(
    columns={"striker": "player", "innings": "bat_innings"})
combined = combined.merge(role_info, on="player", how="left")
combined.loc[combined["batting_impact"] < 10, "role"] = "Bowler"

combined = combined.sort_values("combined_impact", ascending=False).reset_index(drop=True)
combined["rank"] = combined.index + 1

print("Top 20 overall (combined impact):")
print(combined.head(20)[
    ["rank", "player", "batting_impact", "bowling_impact",
     "combined_impact", "role"]
].round(2).to_string(index=False))

Top 20 overall (combined impact):
 rank          player  batting_impact  bowling_impact  combined_impact         role
    1      AD Russell           74.58           55.03           102.10     Finisher
    2       SP Narine           58.30           67.26            96.41       Opener
    3       SR Watson           59.95           56.36            88.13     Finisher
    4      SL Malinga            0.00           83.91            83.91       Bowler
    5        DJ Bravo           48.42           56.14            80.35     Finisher
    6        DW Steyn            0.00           75.48            75.48       Bowler
    7       CH Morris           18.31           66.09            75.24       Bowler
    8     Rashid Khan           18.93           64.95            74.42 Middle-order
    9       H Klaasen           74.18            0.00            74.18     Finisher
   10 Harbhajan Singh           25.37           60.22            72.90 Middle-order
   11       IK Pathan           37.41     

In [16]:
# Add "All-rounder" tag where both contributions are meaningful
def refine_role(row):
    bat, bowl = row["batting_impact"], row["bowling_impact"]
    if bat >= 30 and bowl >= 30:
        return "All-rounder"
    if bat >= 10 and bowl >= 30:
        return "Bowling all-rounder"
    if bat >= 30 and bowl >= 10:
        return "Batting all-rounder"
    return row["role"]

combined["role"] = combined.apply(refine_role, axis=1)

print("Role distribution:")
print(combined["role"].value_counts().to_string())
print()
print("Top 20 with refined roles:")
print(combined.head(20)[
    ["rank", "player", "batting_impact", "bowling_impact",
     "combined_impact", "role"]
].round(2).to_string(index=False))

Role distribution:
role
Bowler                 179
Finisher                51
Opener                  49
Middle-order            23
Batting all-rounder     14
All-rounder             12
Bowling all-rounder      9
Anchor                   4

Top 20 with refined roles:
 rank          player  batting_impact  bowling_impact  combined_impact                role
    1      AD Russell           74.58           55.03           102.10         All-rounder
    2       SP Narine           58.30           67.26            96.41         All-rounder
    3       SR Watson           59.95           56.36            88.13         All-rounder
    4      SL Malinga            0.00           83.91            83.91              Bowler
    5        DJ Bravo           48.42           56.14            80.35         All-rounder
    6        DW Steyn            0.00           75.48            75.48              Bowler
    7       CH Morris           18.31           66.09            75.24 Bowling all-rounder
    

In [17]:
combined.to_csv("../data/processed/player_impact.csv", index=False)
print("Saved: data/processed/player_impact.csv | shape:", combined.shape)
print()

legends = ["V Kohli", "DA Warner", "AB de Villiers", "RG Sharma",
           "MS Dhoni", "KL Rahul", "AD Russell", "JJ Bumrah",
           "SL Malinga", "SP Narine", "HH Pandya", "KA Pollard"]
legend_check = combined[combined["player"].isin(legends)][
    ["rank", "player", "batting_impact", "bowling_impact",
     "combined_impact", "role"]
].round(2).sort_values("rank")

print("Legend sanity check:")
print(legend_check.to_string(index=False))
print()

# Quick distribution check
print("How many players in top 10/25/50/100:")
for cutoff in [10, 25, 50, 100]:
    n = int((combined["rank"] <= cutoff).sum())
    print(f"  Top {cutoff}: {n}")

Saved: data/processed/player_impact.csv | shape: (341, 7)

Legend sanity check:
 rank         player  batting_impact  bowling_impact  combined_impact                role
    1     AD Russell           74.58           55.03           102.10         All-rounder
    2      SP Narine           58.30           67.26            96.41         All-rounder
    4     SL Malinga            0.00           83.91            83.91              Bowler
   12      HH Pandya           54.46           35.12            72.02         All-rounder
   13 AB de Villiers           71.96            0.00            71.96            Finisher
   14      JJ Bumrah            0.00           71.37            71.37              Bowler
   18     KA Pollard           54.81           29.19            69.40 Batting all-rounder
   49       MS Dhoni           60.55            0.00            60.55            Finisher
   58      RG Sharma           55.00            7.07            58.54            Finisher
   95      DA Warner